# Automated GD Modelling POC

## Day 1 — Environment and ReplicatorAgent setup

This notebook records the reproducible setup and smoke tests for the ReplicatorBench-based proof of concept.

Pinned ReplicatorBench commit:

`fb6a804fd710764f3ad3c8b84e1323c2804c4776`


In [1]:
from pathlib import Path
import os
import platform
import subprocess
import sys

POC_ROOT = Path.cwd().resolve()
REPLICATORBENCH_ROOT = Path(
    os.environ.get(
        "REPLICATORBENCH_DIR",
        Path.home() / "llm-benchmarking" / "replicatorbench",
    )
).resolve()

print("POC repository:", POC_ROOT)
print("ReplicatorBench:", REPLICATORBENCH_ROOT)
print("Python:", sys.version)
print("Platform:", platform.platform())


POC repository: /Users/juanlopez/sysrisk-replicator-poc/notebooks
ReplicatorBench: /Users/juanlopez/llm-benchmarking/replicatorbench
Python: 3.11.15 (main, Jun 11 2026, 15:14:57) [Clang 20.1.8 ]
Platform: macOS-26.5.2-arm64-arm-64bit


In [2]:
expected_commit = "fb6a804fd710764f3ad3c8b84e1323c2804c4776"

actual_commit = subprocess.run(
    ["git", "-C", str(REPLICATORBENCH_ROOT.parent), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

assert actual_commit == expected_commit, (
    f"Unexpected ReplicatorBench commit: {actual_commit}"
)

print("Pinned commit verified:", actual_commit)


Pinned commit verified: fb6a804fd710764f3ad3c8b84e1323c2804c4776


In [3]:
subprocess.run(
    ["docker", "info"],
    check=True,
    stdout=subprocess.DEVNULL,
)

print("Docker daemon: reachable")


Docker daemon: reachable


In [4]:
result = subprocess.run(
    ["make", "check-deps"],
    cwd=REPLICATORBENCH_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(result.stdout)


python3 core/check_deps.py pytest pytest_cov openai dotenv pymupdf pyreadr pandas numpy docker docx
All required imports available.



In [5]:
result = subprocess.run(
    ["make", "check-docker"],
    cwd=REPLICATORBENCH_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(result.stdout)


Docker: OK



In [6]:
from pathlib import Path
import subprocess

REPLICATORBENCH_ROOT = (
    Path.home() / "llm-benchmarking" / "replicatorbench"
).resolve()

command = [
    "make",
    "extract-stage1",
    "STUDY=./data/original/1/input",
    "MODEL=gpt-4o",
]

result = subprocess.run(
    command,
    cwd=REPLICATORBENCH_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

assert result.returncode == 0

python3 core/check_deps.py pytest pytest_cov openai dotenv pymupdf pyreadr pandas numpy docker docx
All required imports available.
python -m info_extractor --stage stage_1 --difficulty easy --study-path ./data/original/1/input --model-name gpt-4o
2026-07-19 21:37:15,091 - replication - INFO - Running extraction for ./data/original/1/input at easy difficulty, stage=stage_1


model name for extractor stage: gpt-4o


2026-07-19 21:37:15,092 - replication - INFO - Running Stage 1: original study extraction
PDF is short (8 pages). Returning full text.
=== GENERATED PROMPT (Stage 1) ===
2026-07-19 21:37:15,139 - replication - INFO - === GENERATED PROMPT (Stage 1) ===
You are an information extraction assistant tasked with filling out a structured JSON template based on research documents.

You will be provided with:
1. A JSON template where each key contains a description of what is expected
2. The original paper manuscript (original_paper.pdf)
3. Initial details file (initial_details_easy.

The first run used ./data/original/1 and returned code 0 despite reading no study content, producing an all-“not stated” JSON. After correcting the study path to ./data/original/1/input, the extractor produced substantive claim, data, method, result, and metadata fields. This confirms the extraction stage works, while also exposing a silent-failure risk when the study path is incorrect.

In [ ]:
from pathlib import Path
import subprocess

REPLICATORBENCH_ROOT = (
    Path.home() / "llm-benchmarking" / "replicatorbench"
).resolve()

command = [
    "make",
    "pipeline-easy",
    "STUDY=./data/original/1/input",
    "MODEL=gpt-4o",
]

result = subprocess.run(
    command,
    cwd=REPLICATORBENCH_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

assert result.returncode == 0